In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


c:\Users\Riya Malhotra\Desktop\g\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
import json

In [3]:
with open("product_catalog.json") as f:
    product=json.load(f)

In [4]:
for products in product:
    print(products["name"])

Wireless Headphones
Smart Watch
Mechanical Keyboard
Laptop Stand
USB-C Hub
Wireless Mouse
Portable SSD
Webcam HD
Gaming Monitor
Noise Cancelling Earbuds


In [ ]:
@tool
def search_product(query:str)->str:
    """You are a helpful product assistant for an online tech store.\n"
        "You have access to two data sources:\n\n"
        "1. PRODUCT CATALOG — Use search_products, get_product_details, "
        "compare_products, or list_all_products to look up items from our store.\n"
        "2. WEB SEARCH — Use web_search to find live info from the internet "
        "(current prices, reviews, competitor products, latest news).\n"
        "   Use scrape_web_page to read a specific URL.\n\n"
        "Rules:\n"
        "- Always check our catalog FIRST before searching the web.\n"
        "- Use web search only when the user asks for live/current info.\n"
        "- Provide accurate prices, ratings, and availability.""""
    matches=[p for p in product if query.lower() in str(p).lower()] #matches=[]; for p in product: matches.append(p);
    if not matches :
        return "no products found"
    for m in matches:
        print(m)

In [6]:
# search_product("Laptop Stand") - tool not callable

In [7]:
@tool
def get_product_details(product_name:str)->str:
    """get the details of the product through its name and return them via this function"""
    for p in product:
        if product_name.lower() in p["name"].lower():
            return str(p)
    return f"product not found"
    

In [8]:
@tool
def compare_product(product1:str,product2:str)->str:

    """compare two products based on price, rating, feature"""

    result1="not found"
    result2="not found"
    
    for p in product:
        if product1.lower() in p["name"].lower():
            result1=str(p)
        if product2.lower() in p["name"].lower():
            result2=str(p)
    return f"product1: {result1} \n product2: {result2}"
        

In [9]:
@tool
def list_all_products()->str:
    """list all the products in the catalog"""
    return str(product)

In [10]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [26]:
agent=create_agent(
    llm,
    tools=[search_product,get_product_details,compare_product,list_all_products,web_search,scrape_web_page],
    system_prompt="""you are a helpful product assistant for an online tech store, you have access to all the data in product_catalog.json file. use search_product to search any product using the keyword given as input, use get_product_details for details of specific product, use compare_product to compare two of them given to you and use list_all_products to list all of them""",
    checkpointer=InMemorySaver()
)

In [12]:
q=input("enter your query")

In [13]:
#response=agent.invoke({"messages":{"role":"user","content":q}})
#response

In [14]:
#print(response["messages"][-1].content)

In [15]:
def ask_q(q:str)->str:
    config={"configurable":{"thread_id":"session_1"}}
    result=agent.invoke({"messages":{"role":"user","content":q}},config)
    print(result["messages"][-1].content)

In [18]:
ask_q("what is the price and rating of the Portable SSD")

The price of the Portable SSD is $89.99 and its rating is 4.9.


In [32]:
load_dotenv()
import requests,os
from bs4 import BeautifulSoup
from tavily import TavilyClient

tavily_client=TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [33]:
@tool
def web_search(search_query:str)->str:
    """Search the web for the product information, reviews or prices. Use this when the user wants live/current information that is not in our catalog like latest prices on amazon, new launches, or expert reviews."""
    try:
        response=tavily_client.search(query=search_query, max_results=5)
        results=[]
        for r in response.get("results",[]):
            title=r.get("title","No title")
            url=r.get("url","")
            snippet=r.get("content","")[:200]
            results.append(f"Title:{title}\nURL:{url}\nSnippet:{snippet}")
        return"\n\n---\n\n".join(results if results else "No web results found.")
    except Exception as e:
        return f"Web Search error:{str(e)}"
                           

In [ ]:
@tool
def scrape_web_page(url:str)->str:
    """Scrape a web page and return its text content.Use this to fetch live information from URLs like product pages, review sites, or tech news articles. pass the full URL including https://"""
    try:
        headers={"User-Agent":"Mozilla/5.0 (Educational Agent Bot)"}
        response=requests.get(url,headers=headers,timeout=10)
        response.raise_for_status()

        soup=BeautifulSoup(response.text,"html.parser")

        for tag in soup(["scrpt","style","nav","footer","header"]):
            tag.decompose()

        text=soup.get_text(separator="\n",strip=True)
        return text[:3000] 
    except Exception as e:
        return f"error scraping {url}:{str(e)}"
    

    print("Web tools defined: web_search (Tavily), scrape_web_page")
        
        

In [38]:
ask_q("what is the average price of audio devices nowadays in ruppees?")

The current average price of audio devices in rupees can range from under ₹500 for basic earbuds to over ₹2,00,000 for high-end headphones or speakers, depending on the brand, quality, and features.


In [36]:
ask_q("what is the average price of an apple iphone newly launched?")

The current average price of a newly launched Apple iPhone can range from around $600 for the base model to over $1,500 for the highest-end model, depending on the storage capacity, condition, and location. Please note that prices may vary depending on the region and availability. If you're looking for a specific iPhone model, I can try to provide more detailed pricing information.
